### Gas Cylinder Fine-Tuning

Outputs: training run directory, `best.pt`, training metrics CSV, loss/PR curves.

Datasets:
- `data/gas_bottles/` — CylinDeRS dataset
https://universe.roboflow.com/klearchos-stavrothanasopoulos-konstantinos-gkountakos-6jwgj/cylinders-iaq6n/dataset/3
- `data/gas_cylinder/` — Roboflow gas-cylinder dataset
https://universe.roboflow.com/dl-zepmn/gas-cylinder-fn6bf/dataset/1
- `data\gas_hack` - gas Computer Vision Dataset
https://universe.roboflow.com/hack-rhpxr/gas-bsckk/dataset/1

Each dataset is trained with the same settings so results can be compared. Runs are saved in separate folders so they never overwrite each other.

Outputs per dataset: training run directory, `best.pt`, `last.pt`, `results.csv`, `training_summary.csv`, training plots.

In [11]:
from pathlib import Path
import pandas as pd
import torch
from ultralytics import YOLO
import yaml

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.14.1+cpu
CUDA available: False


In [12]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = ROOT / "data"
RUNS_DIR = ROOT / "results" / "gas_bottle_training"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

device = 0 if torch.cuda.is_available() else "cpu"
print(f"Data directory: {DATA_DIR.relative_to(ROOT)}")
print(f"Runs directory: {RUNS_DIR.relative_to(ROOT)}")
print(f"Device: {device}")

Data directory: data
Runs directory: results\gas_bottle_training
Device: cpu


### Dataset inventory

Quick check of the fine-tuned datasets

In [13]:
DATASET_CANDIDATES = [
    "gas_bottles",
    "gas_cylinder",
    "gas_hack",
]

def load_yaml(ds_dir):
    for name in ["data.yaml", "data_ultralytics.yaml"]:
        p = ds_dir / name
        if p.exists():
            return p, yaml.safe_load(p.read_text())
    return None, None

inventory = []

for name in DATASET_CANDIDATES:
    ds = DATA_DIR / name
    row = {"dataset": name, "exists": ds.exists()}

    if not ds.exists():
        inventory.append(row)
        print(f"{name:15s} MISSING")
        continue

    yaml_path, cfg = load_yaml(ds)
    row["yaml"] = yaml_path.name if yaml_path else None
    row["nc"] = cfg.get("nc") if cfg else None
    row["names"] = str(cfg.get("names")) if cfg else None

    for split in ["train", "valid", "test"]:
        img_dir = ds / split / "images"
        lbl_dir = ds / split / "labels"
        n_img = len(list(img_dir.glob("*"))) if img_dir.exists() else 0
        n_lbl = len(list(lbl_dir.glob("*"))) if lbl_dir.exists() else 0
        row[f"{split}_images"] = n_img
        row[f"{split}_labels"] = n_lbl

    inventory.append(row)
    print(f"{name:15s} exists  nc={row['nc']}  names={row['names']}")
    print(f"{'':15s} train={row['train_images']} valid={row['valid_images']} test={row['test_images']}")

df_inventory = pd.DataFrame(inventory)
INVENTORY_CSV = RUNS_DIR / "dataset_inventory.csv"
df_inventory.to_csv(INVENTORY_CSV, index=False)
print(f"\nInventory saved to {INVENTORY_CSV.relative_to(ROOT)}")

gas_bottles     exists  nc=1  names=['gas_cylinder']
                train=18137 valid=4862 test=2270
gas_cylinder    exists  nc=2  names=['Cap_Close', 'Cap_open']
                train=709 valid=207 test=108
gas_hack        exists  nc=1  names=['gas']
                train=88 valid=24 test=10

Inventory saved to results\gas_bottle_training\dataset_inventory.csv


### Training configuration

Same settings for every dataset so the runs are comparable. Small image size, small batch, a few epochs.

In [14]:
EPOCHS = 5
IMGSZ = 416
BATCH = 4
FRACTION = 0.25
WORKERS = 0
SEED = 0

print(f"epochs={EPOCHS}  imgsz={IMGSZ}  batch={BATCH}  fraction={FRACTION}  workers={WORKERS}")

epochs=5  imgsz=416  batch=4  fraction=0.25  workers=0


### Train on every available dataset

Loops over the datasets that exist on disk. Each run goes into its own folder `results/gas_bottle_training/yolo11n_<dataset>/`.

In [15]:
trained_datasets = []

for name in DATASET_CANDIDATES:
    ds = DATA_DIR / name
    if not ds.exists():
        print(f"[skip] {name} not found")
        continue

    yaml_path, cfg = load_yaml(ds)
    if yaml_path is None:
        print(f"[skip] {name} has no yaml")
        continue

    run_name = f"yolo11n_{name}"
    print("\n" + "=" * 60)
    print(f"Training {name}  ->  run name: {run_name}")
    print(f"nc={cfg.get('nc')}  names={cfg.get('names')}")
    print("=" * 60)

    model = YOLO("yolo11n.pt")

    model.train(
        data=str(yaml_path),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        fraction=FRACTION,
        workers=WORKERS,
        device=device,
        project=str(RUNS_DIR),
        name=run_name,
        exist_ok=True,
        plots=True,
        verbose=True,
        val=True,
        seed=SEED
    )

    trained_datasets.append(name)

print(f"\nFinished training on: {trained_datasets}")


Training gas_bottles  ->  run name: yolo11n_gas_bottles
nc=1  names=['gas_cylinder']
New https://pypi.org/project/ultralytics/8.4.175 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.174  Python-3.11.9 torch-2.14.1+cpu CPU (AMD Ryzen 7 6800HS with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\data\gas_bottles\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.25, freeze=None, hsv_h=0.015, hsv_s=0.

In [16]:
from pathlib import Path
import pandas as pd
import torch
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
RUNS_DIR = ROOT / "results" / "gas_bottle_training"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

DATASET_CANDIDATES = [
    "gas_bottles",
    "gas_cylinder",
    "gas_hack",
]

def load_yaml(ds_dir):
    for name in ["data.yaml", "data_ultralytics.yaml"]:
        p = ds_dir / name
        if p.exists():
            return p, yaml.safe_load(p.read_text())
    return None, None

device = 0 if torch.cuda.is_available() else "cpu"

EPOCHS = 5
IMGSZ = 416
BATCH = 4
FRACTION = 0.25
WORKERS = 0
SEED = 0

print(f"ROOT:   {ROOT}")
print(f"Data:   {DATA_DIR}")
print(f"Runs:   {RUNS_DIR}")
print(f"Device: {device}")

ROOT:   d:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo
Data:   d:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\data
Runs:   d:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\results\gas_bottle_training
Device: cpu


### Collect training summaries

For every dataset that was trained, read the run's `results.csv` and write a one-row `training_summary.csv` with the last-epoch metrics.

These are last-epoch training-run metrics, not necessarily the metrics of `best.pt`. Standalone evaluation of `best.pt` happens in Notebook 4.

In [17]:
def pick(row, *keys, default=None):
    for k in keys:
        for c in row.index:
            if c.lower() == k.lower():
                return row[c]
    return default

summaries = []

for name in DATASET_CANDIDATES:
    run_dir = RUNS_DIR / f"yolo11n_{name}"
    results_csv = run_dir / "results.csv"

    if not results_csv.exists():
        print(f"[skip] {name}: no results.csv at {run_dir}")
        continue

    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    last = df.iloc[-1]

    ds = DATA_DIR / name
    _, cfg = load_yaml(ds)

    row = {
        "dataset": name,
        "classes": str(cfg.get("names")) if cfg else None,
        "nc": cfg.get("nc") if cfg else None,
        "epochs": EPOCHS,
        "imgsz": IMGSZ,
        "batch": BATCH,
        "fraction": FRACTION,
        "device": str(device),
        "run_dir": str(run_dir.relative_to(ROOT)),
        "best_weights": str((run_dir / "weights" / "best.pt").relative_to(ROOT)),
        "precision": pick(last, "metrics/precision(B)", "precision"),
        "recall": pick(last, "metrics/recall(B)", "recall"),
        "mAP50": pick(last, "metrics/mAP50(B)", "mAP50"),
        "mAP50-95": pick(last, "metrics/mAP50-95(B)", "mAP50-95"),
        "train_box_loss": pick(last, "train/box_loss"),
        "train_cls_loss": pick(last, "train/cls_loss"),
        "val_box_loss": pick(last, "val/box_loss"),
        "val_cls_loss": pick(last, "val/cls_loss"),
    }
    summaries.append(row)

    summary_path = run_dir / "training_summary.csv"
    pd.DataFrame([row]).to_csv(summary_path, index=False)
    print(f"{name:15s} precision={row['precision']}  recall={row['recall']}  mAP50={row['mAP50']}")

df_summaries = pd.DataFrame(summaries)
ALL_SUMMARIES_CSV = RUNS_DIR / "all_training_summaries.csv"
df_summaries.to_csv(ALL_SUMMARIES_CSV, index=False)
print(f"\nCombined summary saved to {ALL_SUMMARIES_CSV.relative_to(ROOT)}")

gas_bottles     precision=0.98125  recall=0.76173  mAP50=0.87258
gas_cylinder    precision=0.78941  recall=0.31081  mAP50=0.34501
gas_hack        precision=0.00333  recall=1.0  mAP50=0.8358

Combined summary saved to results\gas_bottle_training\all_training_summaries.csv


### Overview

Side-by-side view of the last-epoch metrics for every trained dataset.

In [18]:
if not df_summaries.empty:
    cols = ["dataset", "classes", "nc", "precision", "recall", "mAP50", "mAP50-95"]
    print(df_summaries[cols].to_string(index=False))
else:
    print("No training runs to show.")

     dataset                   classes  nc  precision  recall   mAP50  mAP50-95
 gas_bottles          ['gas_cylinder']   1    0.98125 0.76173 0.87258   0.83559
gas_cylinder ['Cap_Close', 'Cap_open']   2    0.78941 0.31081 0.34501   0.12624
    gas_hack                   ['gas']   1    0.00333 1.00000 0.83580   0.59004


### Verify saved weights

Confirm `best.pt` and `last.pt` exist for every trained dataset. Notebook 4 loads these files for standalone evaluation.

In [19]:
for name in DATASET_CANDIDATES:
    run_dir = RUNS_DIR / f"yolo11n_{name}"
    best = run_dir / "weights" / "best.pt"
    last = run_dir / "weights" / "last.pt"
    print(f"{name:15s} best={best.exists()}  last={last.exists()}  -> {run_dir.relative_to(ROOT)}")

gas_bottles     best=True  last=True  -> results\gas_bottle_training\yolo11n_gas_bottles
gas_cylinder    best=True  last=True  -> results\gas_bottle_training\yolo11n_gas_cylinder
gas_hack        best=True  last=True  -> results\gas_bottle_training\yolo11n_gas_hack
